# Absolute Effectiveness
Compute habitat condition and loss metrics for a selected site.

In [ ]:
# Select site by site_id (from the active SITE_GROUP's cleaned sites, SITE_GEOMS)
site_id = 11116292

In [ ]:
from pathlib import Path
import os
import sys
import ee
import geemap

cur = Path.cwd().resolve()
for parent in [cur] + list(cur.parents):
    if parent.name == "tpae":
        os.chdir(parent)
        break

sys.path.insert(0, str((Path.cwd() / "src").resolve()))

from utils.variables import (
    PROJECT,
    ANALYSIS_END_YR,
    DRIVER_LABELS,
    KERNEL_RADIUS_M,
)

from sites.site_selector import bounds_rectangle, ee_geometry, load_sites, site_region
from absolute_effectiveness.absolute_effectiveness import (
    DRIVER_COLUMNS,
    N_HABITATS_LOST,
    analyze_site,
)
from absolute_effectiveness.data_processor import DataProcessor
from absolute_effectiveness.habitat_condition import HabitatConditionAnalyzer
from absolute_effectiveness.habitat_loss import HabitatLossAnalyzer
from absolute_effectiveness.visualization import VisualizationService

ee.Authenticate()
ee.Initialize(project=PROJECT)

processor = DataProcessor.from_gee_defaults()
condition_analyzer = HabitatConditionAnalyzer()
loss_analyzer = HabitatLossAnalyzer()
visualization_service = VisualizationService()

In [ ]:
# Get site-specific information from the cleaned sites
sites = load_sites().set_index("site_id")
START_YR = int(sites.loc[site_id, "start_yr"])
site_geom = ee_geometry(sites.geometry[site_id])
# Site polygons, for clipping map layers
site_polygons, _ = site_region(sites.geometry[site_id])
# Input images are read within the site and the intactness kernel's reach
inputs_region = bounds_rectangle(sites.geometry[site_id].bounds, KERNEL_RADIUS_M)

In [ ]:
# Calculate the site's results exactly as absolute_effectiveness.py does
row = analyze_site(site_id, START_YR, sites.geometry[site_id])
if row.get("error"):
    raise RuntimeError(f"Site {site_id} failed: {row['error']}")

# Build the rasters shown in the maps below, with the same steps
GLC_processed = processor.process_glc(inputs_region, START_YR)
GPW_processed = processor.process_gpw(START_YR)
FORTY_processed = processor.process_forty(inputs_region, START_YR)
HGFC_processed = processor.process_hgfc(START_YR)
land_mask = processor.get_land_mask()
lc_end = condition_analyzer.get_land_cover(
    GLC_processed, GPW_processed, FORTY_processed, ANALYSIS_END_YR
)
habitat_binary_end = condition_analyzer.get_habitat_binary_end(lc_end, HGFC_processed)
lc_start = condition_analyzer.get_land_cover(
    GLC_processed, GPW_processed, FORTY_processed, START_YR
)
habitat_binary_start = condition_analyzer.get_habitat_binary_start(lc_start)
habitat_loss_raster = loss_analyzer.get_habitat_loss_raster(
    habitat_binary_start, habitat_binary_end
)
intactness_raster = condition_analyzer.get_intactness_raster(
    habitat_binary_end, condition_analyzer.build_kernel(), land_mask
)
driver_class = loss_analyzer.get_driver_class_image(lc_end, habitat_loss_raster)
habitat_class = loss_analyzer.get_habitat_class_image(lc_start, habitat_loss_raster)

# Create Sentinel-2 composites (for visualization purposes only)
start_s2_composite = visualization_service.get_s2_med_composite(site_geom, START_YR)
end_s2_composite = visualization_service.get_s2_med_composite(
    site_geom, ANALYSIS_END_YR
)

# Print all results
print(f"Site ID: {site_id}")
print(f"Analysis Period: {START_YR} - {ANALYSIS_END_YR}")
print(f"\nHabitat Extent Score: {row['extent']:.2f}")
print(f"Habitat Intactness Score: {row['intactness']:.2f}")
print(f"\nHabitat Condition Score: {row['condition']:.2f}")
print(f"Habitat Loss Score: {row['loss']:.2f}")
print("\nOverall habitat loss:")
loss_analyzer.print_habitat_loss_metrics(
    "Total",
    row["loss_area_km2"],
    row["loss_pct_habitat"],
    pct_site=row["loss_pct_site"],
)
print("\nDrivers of habitat loss:")
for class_id, driver in sorted(
    DRIVER_COLUMNS.items(), key=lambda item: row[f"{item[1]}_area_km2"], reverse=True
):
    if row[f"{driver}_area_km2"]:
        loss_analyzer.print_habitat_loss_metrics(
            DRIVER_LABELS[class_id],
            row[f"{driver}_area_km2"],
            row[f"{driver}_pct_habitat"],
            pct_site=row[f"{driver}_pct_site"],
        )
print(f"\nTop {N_HABITATS_LOST} types of habitat lost:")
for i in range(1, N_HABITATS_LOST + 1):
    label = row.get(f"habitat_lost_{i}")
    if label:
        loss_analyzer.print_habitat_loss_metrics(
            label,
            row[f"habitat_lost_{i}_area_km2"],
            row[f"habitat_lost_{i}_pct_start"],
            reference_label=f"of site {label.lower()} at start year",
        )

## Visualization

In [ ]:
from utils.variables import FORTY_PALETTE, GLC_PALETTE, INTACTNESS_SCALE_M

# Sentinel-2 RGB viz parameters
s2_rgb_viz = {
    "min": 0.0,
    "max": 0.3,
    "bands": ["B4", "B3", "B2"],
}

Map = geemap.Map(height=900)
Map.add_basemap("Esri.WorldGrayCanvas")
Map.add_basemap("Esri.WorldImagery")

# Map.addLayer(start_s2_composite, s2_rgb_viz, "S2, Start Year", 0)
# Map.addLayer(end_s2_composite, s2_rgb_viz, "S2, End Year", 0)
Map.addLayer(
    GLC_processed.select("GLC_2022"),
    {"min": 0, "max": 36, "palette": GLC_PALETTE},
    "Global Land Cover",
    0,
)
Map.addLayer(
    GPW_processed.select("GPW_2022").selfMask(),
    {"min": 1, "max": 2, "palette": ["#ffcd73", "#ff9916"]},
    "Global Pasture Watch",
    0,
)
Map.addLayer(
    FORTY_processed.select(f"FORTY_{ANALYSIS_END_YR}"),
    {"min": 1, "max": 7, "palette": FORTY_PALETTE},
    "ForTy",
    0,
)
Map.addLayer(
    HGFC_processed.selfMask(),
    {
        "min": START_YR - 2000,
        "max": ANALYSIS_END_YR - 2000,
        "palette": ["yellow", "red"],
    },
    "Hansen Global Forest Change",
    0,
)
Map.addLayer(site_geom, {"color": "white"}, "Site", 1, 0.5)
Map.addLayer(
    habitat_binary_end.selfMask().clipToCollection(site_polygons),
    {"palette": ["#1a9850"]},
    "Habitat Extent",
)
Map.addLayer(
    intactness_raster.clipToCollection(site_polygons).reproject(
        crs=habitat_binary_end.projection(), scale=INTACTNESS_SCALE_M
    ),
    {
        "min": 0,
        "max": 1,
        "palette": ["#d73027", "#fdae61", "#fee08b", "#d9ef8b", "#1a9850"],
    },
    "Habitat Intactness",
)
Map.addLayer(
    habitat_binary_start.selfMask().clipToCollection(site_polygons),
    {"palette": "#1a9850"},
    "Habitat at start year",
)
Map.addLayer(
    habitat_loss_raster.selfMask().clipToCollection(site_polygons),
    {"palette": ["red"]},
    "Habitat Loss",
)

Map.addLayer(
    driver_class.selfMask().clipToCollection(site_polygons),
    {"min": 1, "max": 5, "palette": ["yellow", "red", "orange", "green", "#c51b7d"]},
    "Drivers of habitat loss",
    0,
)
Map.addLayer(
    habitat_class.selfMask().clipToCollection(site_polygons),
    {"palette": GLC_PALETTE},
    "Types of habitat lost",
    0,
)

Map.centerObject(site_geom)

Map